# exp_021 · Summary — Cross-Level Metrics

Aggregates geometric signals from Levels 1–3 and compares them against
ground-truth annotations for all 10 samples.

---

### Column definitions

| Column | Source | What it measures |
|--------|--------|-----------------|
| `strength` | Level 1 `curvature_z0` | max/mean curvature ratio. > 1 = peak exists; higher = sharper cut. |
| `angular_min` | Level 1 `angular_z0` | Most negative cosine. < 0 = direction reversal. More negative = harder cut. |
| `gt_s` | Manual annotation | Ground-truth transition time in seconds. |
| `gt_p` | Converted from gt_s | Ground-truth latent frame (float, sub-frame precision). |
| `commit_step` | Level 2 `dissolve_by_step` | Earliest denoising step τ at which the curvature-peak location stabilises. Lower = more decisive. |
| `pred_peak_p` | Level 2 `pred_mag` | Frame with highest mean velocity across all 40 steps. |

---

> **Class guide:** 🔵 C1 (easiest) · 🟢 C2 · 🟠 C5 · 🔴 C6 · 🟣 C8 (hardest)


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))  # find trajectory_utils.py

from trajectory_utils import *

# Load all 10 samples (extracts features, frees raw tensors)
records = load_all_records()

# Load manual GT transition annotations
gt_dict = load_gt_annotations()

n_gt = sum(1 for v in gt_dict.values() if v is not None)
print(f"\n✓ GT annotations available for {n_gt}/{len(gt_dict)} samples")
for sid, gs in sorted(gt_dict.items()):
    gp_str = f"  p≈{gt_latent(sid, gt_dict):.1f}" if gs is not None else "  —"
    gs_str = f"{gs:.2f}s" if gs is not None else " —"
    print(f"  {sid:<50s}  {gs_str:>7s}{gp_str}")


In [ ]:
def _commit_step_fn(dissolve_by_step: np.ndarray, tolerance: int = 1) -> int:
    final = dissolve_by_step[-1]
    for i in range(len(dissolve_by_step)):
        if np.all(np.abs(dissolve_by_step[i:] - final) <= tolerance):
            return int(i)
    return len(dissolve_by_step) - 1

rows = []
for r in records:
    f         = r["feats"]
    pm        = f["pred_mag"]
    mean_pm   = pm.mean(axis=0)
    pred_peak = int(np.argmax(mean_pm))
    commit    = _commit_step_fn(f["dissolve_by_step"])
    gs        = gt_dict.get(r["sample_id"])
    gp_approx = round(gt_latent(r["sample_id"], gt_dict), 1) if gs is not None else None

    rows.append({
        "sample":       r["short_id"],
        "class":        r["cls"],
        "cls_label":    r["cls_label"],
        "gt_s":         round(gs, 2) if gs is not None else None,
        "gt_p":         gp_approx,
        "strength":     round(f["dissolve_z0_strength"] if "dissolve_z0_strength" in f
                              else float(f["curvature_z0"].max() /
                                         (f["curvature_z0"].mean() + 1e-8)), 2),
        "angular_min":  round(float(f["angular_z0"].min()), 3),
        "pred_peak_p":  pred_peak,
        "commit_step":  commit,
    })

df_table = pd.DataFrame(rows).sort_values(["class", "sample"]).reset_index(drop=True)

def _cls_color(cls_str):
    return CLASS_COLORS.get(str(cls_str), "#888")

def _row_style(row):
    col = _cls_color(str(row["class"]))
    return [f"background-color:{col}18; border-left:4px solid {col}"] * len(row)

styled = (
    df_table.style
    .apply(_row_style, axis=1)
    .background_gradient(subset=["strength"],    cmap="YlOrRd", vmin=1, vmax=5)
    .background_gradient(subset=["angular_min"], cmap="RdYlGn", vmin=-1, vmax=1)
    .background_gradient(subset=["commit_step"], cmap="PuBu",   vmin=0, vmax=39)
    .format({
        "strength":    "{:.2f}×",
        "angular_min": "{:.3f}",
        "gt_s":        lambda x: f"{x:.2f}s" if x is not None else "—",
        "gt_p":        lambda x: f"p≈{x:.1f}" if x is not None else "—",
    })
    .set_caption("Geometric Dissolve Metrics — all 10 samples  |  gt_s = manual GT annotation")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "13px"), ("font-weight", "bold"),
                                          ("text-align", "left"), ("margin-bottom", "8px")]},
        {"selector": "th", "props": [("background-color", "#2c2c2c"), ("color", "white"),
                                     ("font-size", "11px"), ("padding", "5px 10px")]},
        {"selector": "td", "props": [("font-size", "11px"), ("padding", "4px 10px")]},
    ])
)
display(styled)

# Class-level aggregation
agg = df_table.groupby("class").agg(
    n=("sample", "count"),
    mean_strength=("strength", "mean"),
    mean_angular_min=("angular_min", "mean"),
    mean_commit_step=("commit_step", "mean"),
).round(3)
print("\nClass-level aggregation:")
display(agg)


---
## Summary Charts

**Left — Dissolve Strength vs. Direction Reversal (scatter)**

Each dot = one sample, coloured by class.

- **Upper-left** (high strength + very negative angular_min) = hard, abrupt dissolve → expected for class 6/8
- **Lower-right** (low strength + near-zero angular_min) = smooth or absent transition → expected for class 1
- Diagonal trend = the two signals agree — the cut is geometrically clear

**Right — Commitment Step by Sample (bar chart)**

y-axis = denoising step τ at which the curvature-peak location stabilises.

- **Low bar** (τ < 10) = model decided the transition location early — confident, clear conditioning
- **High bar** (τ > 30) = late commitment — ambiguous input pair


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "Summary — Geometric Dissolve Signals by Semantic Class  |  🟢=GT available",
    fontsize=11, fontweight="bold"
)

# scatter: strength vs angular_min
ax = axes[0]
for r in records:
    f   = r["feats"]
    col = r["color"]
    sid = r["sample_id"]
    strength    = float(f["curvature_z0"].max() / (f["curvature_z0"].mean() + 1e-8))
    angular_min = float(f["angular_z0"].min())
    ax.scatter(strength, angular_min, color=col, s=110, zorder=3, alpha=0.9,
               label=r["short_id"])
    ax.annotate(f"C{r['cls']}", (strength, angular_min),
                fontsize=8, ha="center", va="center", color="white", fontweight="bold")
    gs = gt_dict.get(sid)
    if gs is not None:
        ax.scatter(strength, angular_min, s=260, facecolors="none",
                   edgecolors="#4CAF50", linewidths=2.0, zorder=4)
ax.axhline(0, color="gray", lw=0.8, ls=":")
ax.axvline(1, color="gray", lw=0.8, ls=":")
ax.set_xlabel("dissolve strength  (max/mean curvature z₀)", fontsize=9)
ax.set_ylabel("angular_min  (most negative cosine)", fontsize=9)
ax.set_title("Curvature Strength vs. Direction Reversal\n(green ring = GT annotation available)", fontsize=9)
ax.legend(fontsize=6, ncol=1, loc="upper left", framealpha=0.8)

# bar: commit_step by sample
ax = axes[1]
commit_vals = [_commit_step_fn(r["feats"]["dissolve_by_step"]) for r in records]
ax.bar(range(len(records)), commit_vals,
       color=[r["color"] for r in records], alpha=0.85)
ax.set_xticks(range(len(records)))
ax.set_xticklabels(
    [r["short_id"].replace("[C", "C").split("]")[0] + "]" for r in records],
    rotation=45, ha="right", fontsize=7
)
ax.set_ylabel("commit step τ", fontsize=9)
ax.set_title("Denoising Step of Curvature-Peak Stabilisation\n(lower = model commits earlier)", fontsize=9)
ax.set_ylim(0, S_STEPS)
ax.axhline(S_STEPS/2, color="gray", lw=0.8, ls=":")

plt.tight_layout()
plt.show()


---
## Final View — Video + Velocity Heatmap Per Sample

Each row: the generated video alongside its `pred_mag` heatmap (velocity magnitude τ × p).

**How to cross-validate:**
1. Play the video and note the second where the visual transition occurs.
2. Convert: `p ≈ floor(time_s × 24 / 8)`.
3. Check if the **green GT line** aligns with the bright vertical stripe in the heatmap.

**If GT line ≈ bright stripe** → `pred_mag` and ground-truth agree.  
**If they disagree** → possible causes: gradual blend (not a hard cut), GT annotation
is at a perceptually ambiguous point, or the model placed the transition differently
from what was perceptually expected.


In [ ]:
import io

all_html = []
all_html.append(
    '<h3 style="font-family:sans-serif">Final View — Video + pred_mag heatmap (per sample)</h3>'
    '<p style="font-family:sans-serif;font-size:12px">'
    'Each row: 🎬 generated video  |  pred_mag (τ × p) heatmap.<br>'
    '🟢 solid=GT annotation  |  cyan=start-clip  magenta=end-clip</p>'
)

for r in records:
    f    = r["feats"]
    pm   = f["pred_mag"]
    col  = r["color"]
    sid  = r["sample_id"]
    vmax = float(np.percentile(pm, 99))
    gs   = gt_dict.get(sid)
    gp   = gt_latent(sid, gt_dict)

    fig, ax = plt.subplots(figsize=(7, 2.8))
    im = ax.imshow(pm, aspect="auto", cmap="plasma", vmin=0, vmax=vmax,
                   origin="upper", interpolation="nearest",
                   extent=[-0.5, f["F"]-0.5, f["S"]-0.5, -0.5])
    ax.axvline(K_LAT   - 0.5, color="#00BCD4", lw=1.2, ls="--", alpha=0.8)
    ax.axvline(END_IDX - 0.5, color="#E91E63", lw=1.2, ls="--", alpha=0.8)
    if gp is not None:
        ax.axvline(gp, color="#4CAF50", lw=2.2, ls="-",
                   label=f"GT {gs:.2f}s (p≈{gp:.1f})", alpha=0.9)
        ax.legend(fontsize=7, loc="upper right", framealpha=0.8)
    ax.set_xlabel("frame p", fontsize=8)
    ax.set_ylabel("step τ", fontsize=8)
    ax.set_xticks(range(f["F"]))
    ax.tick_params(labelsize=6.5)
    strength    = float(f["curvature_z0"].max() / (f["curvature_z0"].mean() + 1e-8))
    angular_min = float(f["angular_z0"].min())
    ax.set_title(
        f"pred_mag  [strength={strength:.1f}×  angular_min={angular_min:.3f}]",
        fontsize=8.5
    )
    plt.colorbar(im, ax=ax, shrink=0.9, pad=0.02)
    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    img_b64 = __import__("base64").b64encode(buf.read()).decode()

    vid_h  = video_html(r["video_path"], width=340, label=r["short_id"], cls=r["cls"])
    plot_h = f'<img src="data:image/png;base64,{img_b64}" style="height:215px;vertical-align:top">'
    all_html.append(
        f'<div style="display:flex;align-items:flex-start;gap:12px;'
        f'margin-bottom:12px;border-bottom:1px solid #ddd;padding-bottom:10px">'
        f'{vid_h}{plot_h}</div>'
    )

display(HTML("\n".join(all_html)))


---
## Next Steps & Research Directions

### Transition time control (highest ROI)
1. **`end_idx` ablation** — Change `end_clip_index` in the config to `{6, 8, 10, 12}`.  
   Hypothesis: the velocity stripe in `pred_mag` shifts with it.
2. **`z_t` mid-trajectory intervention** — Run Stage-1 to the commit step τ*.  
   Then blend `z_t[:,:,p_target,:,:]` toward a shifted position. Continue denoising.  
   Hypothesis: transition shifts by the same amount.

### Style control (medium ROI)
3. **Guidance scale schedule** — Instead of constant `guidance_scale=4.0`, ramp it up
   at the (τ, p) region where `pred_mag` peaks.  
   Hypothesis: amplifies text conditioning exactly where the model generates the transition.
4. **Hidden-state probing** — Train a linear probe to predict "frame is in transition region"
   from Level 3 hidden states.  
   This quantifies which transformer block encodes transition structure.

### Immediate validation with this notebook
- Does `strength` correlate with semantic class? (summary table + scatter)
- Does `commit_step` differ between easy (class 1) and hard (class 8) transitions?
- In the PCA plots: does the GT frame appear as a trajectory kink or cluster boundary?
- In the similarity matrix: do class 6/8 show sharper block structure than class 1?
- Cross-validate: play each video and note the perceptual cut time. Does it match `gt_s`?
